# TerraScope — Satellite Change Intelligence Platform
## Research-Grade Multi-Architecture GPU Training & Benchmarking Pipeline

This notebook orchestrates the reproducible training and quantitative benchmarking of three bi-temporal change detection architectures on Google Colab:
1. **Siamese U-Net**: Convolutional baseline with spatial skip connections and difference feature fusion.
2. **Siamese Vision Transformer (ViT)**: Pure self-attention token-based architecture.
3. **Siamese Swin Transformer**: Shifted-window hierarchical vision transformer with multi-scale difference fusion.

### Benchmark Datasets:
- **LEVIR-CD**: Ultra-high-resolution bi-temporal optical building change detection (RGB).
- **OSCD**: Onera Satellite Change Detection multi-spectral Sentinel-2 dataset.

---
### Reliability & Resilience Safeguards:
- **Periodic Drive Checkpoint Backup**: Checkpoints (`latest.pt` and `best.pt`) are mirrored to Google Drive **at the end of every epoch**, protecting against Colab session timeouts.
- **Accidental Overwrite Protection**: Completed runs are skipped automatically unless `FORCE_RETRAIN = True`.
- **Clean Resume vs. Fresh Retrain**: When `FORCE_RETRAIN = True`, existing checkpoints are ignored to start a fresh run; when `False`, interrupted runs resume seamlessly from the latest epoch.
- **Comprehensive State Recovery**: Checkpoint resume restores model weights, optimizer state, learning rate scheduler state, AMP scaler state (if present), and epoch counters.
- **Pre-Flight Validation Gates**: Dataset structure, label alignments, synthetic CUDA execution, and real-dataloader batch passes must all succeed before long training begins.
- **Missing-Model Safeguard**: Evaluation requires all three architecture checkpoints to be present before running benchmarks, preventing incomplete or misleading comparisons.


### Step 1: Mount Google Drive & Setup Persistent Storage
Mount Google Drive and create persistent folders for checkpoints, experiment logs, and visual analytics.


In [ ]:
from google.colab import drive
import os
import sys
import shutil
from pathlib import Path

drive.mount("/content/drive")

# Persistent storage on Google Drive
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/TerraScope_Outputs")
DRIVE_CHECKPOINTS_DIR = DRIVE_OUTPUT_DIR / "checkpoints"
DRIVE_EXPERIMENTS_DIR = DRIVE_OUTPUT_DIR / "experiments"
DRIVE_VIZ_DIR = DRIVE_OUTPUT_DIR / "visualizations"

for p in (DRIVE_CHECKPOINTS_DIR, DRIVE_EXPERIMENTS_DIR, DRIVE_VIZ_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f" Persistent Google Drive storage verified at: {DRIVE_OUTPUT_DIR}")


### Step 2: GPU Accelerator & CUDA Environment Audit
Inspects GPU hardware and provides realistic, tiered memory advice for batch size selection.


In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL: CUDA GPU is not available! \n"
        "Go to Runtime > Change runtime type > Hardware accelerator > select T4 GPU (or A100/V100)."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("--- GPU Accelerator Hardware Audit ---")
print(f" PyTorch Version : {torch.__version__}")
print(f" CUDA Device     : {gpu_name}")
print(f" Total VRAM      : {total_vram_gb:.2f} GB")

if total_vram_gb >= 14.0:
    print(" VRAM Capacity: High (>=14 GB). Default batch size 8 is recommended for all architectures.")
elif total_vram_gb >= 10.0:
    print(" VRAM Capacity: Moderate (10-14 GB). Batch size 8 is supported for Siamese U-Net and ViT.")
    print("   If Swin Transformer experiences CUDA Out of Memory (OOM), reduce its batch size to 4.")
else:
    print(f" WARNING: Low VRAM ({total_vram_gb:.1f} GB < 10 GB).")
    print("   Batch size 8 will likely cause CUDA Out of Memory (OOM) on Vision Transformers.")
    print("   Mitigation: Change batch_size to 4 (or 2 for Swin) in the EXPERIMENT_MATRIX below.")


### Step 3: Setup Project Workspace
Copies repository code to local fast SSD (`/content/satellite`) for optimal disk throughput while keeping persistent outputs linked to Google Drive.


In [ ]:
DRIVE_PROJECT_PATH = Path("/content/drive/MyDrive/satellite")
LOCAL_PROJECT_PATH = Path("/content/satellite")

if DRIVE_PROJECT_PATH.exists():
    if not LOCAL_PROJECT_PATH.exists():
        print(f"Copying project from {DRIVE_PROJECT_PATH} to {LOCAL_PROJECT_PATH} for SSD throughput...")
        shutil.copytree(str(DRIVE_PROJECT_PATH), str(LOCAL_PROJECT_PATH), dirs_exist_ok=True)
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
elif LOCAL_PROJECT_PATH.exists():
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
else:
    PROJECT_ROOT = Path(os.getcwd())

print(f" Active Project Root: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Step 4: Install Dependencies & Verify Core Modules
Installs required geospatial and deep learning packages (`rasterio`, `geopandas`, `timm`, `reportlab`).


In [ ]:
!pip install -q rasterio geopandas timm reportlab python-dotenv

# Verify crucial imports
import rasterio
import timm
from ml.models import get_model
from ml.utils.device import detect_device

print(f" Active Compute Device: {detect_device()}")
print(" Core dependencies verified successfully.")


### Step 5: Dataset Path Configuration & Structural Integrity Gate
Validates dataset directories before training.
For **OSCD**, points to the `onera DATA` root folder containing:
- `Onera Satellite Change Detection dataset - Images`
- `Onera Satellite Change Detection dataset - Train Labels`
- `Onera Satellite Change Detection dataset - Test Labels`

Execution halts with an informative error if required folders or splits are missing.


In [ ]:
# --- CONFIGURE YOUR DATASET PATHS ON GOOGLE DRIVE ---
LEVIR_CD_DIR = Path("/content/drive/MyDrive/data/LEVIR-CD")
# Point to the root onera DATA folder that contains Images, Train Labels, and Test Labels
OSCD_DIR = Path("/content/drive/MyDrive/data/onera DATA")

def validate_datasets(levir_path: Path, oscd_path: Path) -> dict:
    status = {"levir_ok": False, "oscd_ok": False}
    
    # 1. Validate LEVIR-CD
    if not levir_path.exists():
        print(f" Note: LEVIR-CD path not found at: {levir_path}")
    else:
        missing_levir = []
        for sp in ("train", "val", "test"):
            for sub in ("A", "B", "label"):
                d = levir_path / sp / sub
                if not d.is_dir() or len(list(d.glob("*.*"))) == 0:
                    missing_levir.append(str(d))
        if missing_levir:
            raise FileNotFoundError(
                f"LEVIR-CD missing required subdirectories or image tiles:\n" + "\n".join(missing_levir)
            )
        status["levir_ok"] = True
        print(f" LEVIR-CD validated successfully: {levir_path}")

    # 2. Validate OSCD
    if not oscd_path.exists():
        print(f" Note: OSCD path not found at: {oscd_path}")
    else:
        from ml.datasets.oscd import _find_images_base, _find_label_roots
        img_base = _find_images_base(oscd_path)
        train_lbls = _find_label_roots(oscd_path, explicit_dir=None, category="train")
        test_lbls = _find_label_roots(oscd_path, explicit_dir=None, category="test")
        
        if not img_base.is_dir():
            raise FileNotFoundError(f"OSCD Images folder not found under {oscd_path}")
        if not train_lbls:
            raise FileNotFoundError(f"OSCD Train Labels folder not found under {oscd_path}")
        if not test_lbls:
            raise FileNotFoundError(f"OSCD Test Labels folder not found under {oscd_path}")
        
        status["oscd_ok"] = True
        print(f" OSCD validated successfully:")
        print(f"   Root Dir    : {oscd_path.name}")
        print(f"   Images Base : {img_base.name}")
        print(f"   Train Labels: {train_lbls[0].name}")
        print(f"   Test Labels : {test_lbls[0].name}")

    if not status["levir_ok"] and not status["oscd_ok"]:
        raise SystemExit("CRITICAL: Neither LEVIR-CD nor OSCD was found at the configured paths. Stopping.")
    return status

dataset_status = validate_datasets(LEVIR_CD_DIR, OSCD_DIR)


### Step 6: Pre-Flight GPU Smoke Test & Real-DataLoader Validation
Performs two rapid validation gates:
- **Part A (In-Memory GPU Mechanics)**: Verifies CUDA forward pass, BCE+Dice loss calculation, backward gradients, optimizer step, and checkpoint save/reload integrity.
- **Part B (Real-DataLoader Integration)**: Fetches a single mini-batch from the real LEVIR-CD and OSCD datasets and verifies that all three architectures process real satellite input dimensions and channels without errors.

Execution stops immediately if any check fails.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ml.models import get_model
from ml.training.losses import get_loss_function
from ml.training.optimizer import create_optimizer
from ml.training.callbacks import ModelCheckpointManager, load_checkpoint
from ml.datasets import LEVIRCDDataset, OSCDDataset

print("=== GATELIGHT 1: IN-MEMORY GPU MECHANICS SMOKE TEST ===")
device = torch.device("cuda")

try:
    # 1. Forward pass
    smoke_model = get_model("siamese_unet", in_channels=3, img_size=256, pretrained=False).to(device)
    smoke_model.train()
    t1 = torch.randn(2, 3, 256, 256, device=device)
    t2 = torch.randn(2, 3, 256, 256, device=device)
    target = torch.randint(0, 2, (2, 256, 256), device=device, dtype=torch.float32)
    
    logits = smoke_model(t1, t2)
    assert logits.shape == (2, 1, 256, 256), f"Unexpected output shape: {logits.shape}"
    
    # 2. Loss & Backward
    criterion = get_loss_function("bce_dice")
    loss = criterion(logits, target)
    assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"
    
    optimizer = create_optimizer(smoke_model, name="adamw", lr=3e-4)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 3. Checkpoint save & reload
    temp_ckpt_dir = Path("/tmp/smoke_checkpoints")
    ckpt_mgr = ModelCheckpointManager(temp_ckpt_dir, "smoke_unet", "test_ds")
    saved_path = ckpt_mgr.save(smoke_model, optimizer, None, epoch=1, best_metric=0.85, config={}, is_best=True)
    assert saved_path.is_file(), "Smoke checkpoint creation failed!"
    
    reload_meta = load_checkpoint(saved_path, smoke_model, optimizer, map_location=device)
    assert reload_meta.get("epoch") == 1, "Checkpoint reload failed!"
    print(" In-memory GPU mechanics: PASSED (Forward, Loss, Backward, Optimizer, Checkpoint).")
except Exception as e:
    raise SystemExit(f"CRITICAL: In-memory GPU smoke test failed: {e}")

print("\n=== GATELIGHT 2: REAL DATASET BATCH & ARCHITECTURE VALIDATION ===")
architectures = ("siamese_unet", "vit", "swin")

# Test LEVIR-CD real batch if available
if dataset_status["levir_ok"]:
    print("Testing real LEVIR-CD batch against all 3 architectures...")
    levir_ds = LEVIRCDDataset(LEVIR_CD_DIR, split="train", patch_size=256, stride=256)
    levir_loader = DataLoader(levir_ds, batch_size=2, shuffle=False)
    levir_batch = next(iter(levir_loader))
    b_t1 = levir_batch["before"].to(device)
    b_t2 = levir_batch["after"].to(device)
    b_mask = levir_batch["mask"].to(device)
    assert b_t1.shape == (2, 3, 256, 256), f"LEVIR-CD unexpected batch shape: {b_t1.shape}"

    for arch in architectures:
        m = get_model(arch, in_channels=3, img_size=256, pretrained=False).to(device)
        m.eval()
        with torch.no_grad():
            out = m(b_t1, b_t2)
            assert out.shape == (2, 1, 256, 256), f"{arch} failed on LEVIR-CD: output shape {out.shape}"
        print(f"   {arch:<15} on LEVIR-CD real batch: OK")

# Test OSCD real batch if available
if dataset_status["oscd_ok"]:
    print("Testing real OSCD Sentinel-2 batch against all 3 architectures...")
    oscd_ds = OSCDDataset(OSCD_DIR, split="train", bands=("B04", "B03", "B02"), patch_size=256, stride=256)
    oscd_loader = DataLoader(oscd_ds, batch_size=2, shuffle=False)
    oscd_batch = next(iter(oscd_loader))
    o_t1 = oscd_batch["before"].to(device)
    o_t2 = oscd_batch["after"].to(device)
    o_mask = oscd_batch["mask"].to(device)
    assert o_t1.shape == (2, 3, 256, 256), f"OSCD unexpected batch shape: {o_t1.shape}"

    for arch in architectures:
        m = get_model(arch, in_channels=3, img_size=256, pretrained=False).to(device)
        m.eval()
        with torch.no_grad():
            out = m(o_t1, o_t2)
            assert out.shape == (2, 1, 256, 256), f"{arch} failed on OSCD: output shape {out.shape}"
        print(f"   {arch:<15} on OSCD real batch: OK")

print("=== ALL PRE-FLIGHT VERIFICATION GATES PASSED CLEANLY ===")


## Step 7: Systematic Model Training Pipeline

### Controlled Research Configuration:
Every architecture is trained with strictly identical optimization parameters:
- **Optimizer**: `AdamW` (`weight_decay=1e-4`)
- **Loss Function**: Combined Binary Cross-Entropy + Dice Loss (`bce_dice`)
- **Learning Rate**: $3 \times 10^{-4}$ with Cosine Annealing (`--scheduler cosine`)
- **Batch Size**: 8 (if VRAM is below 10 GB, change `batch_size: 4` in `EXPERIMENT_MATRIX`)
- **Patch Resolution**: $256 \times 256$ pixels
- **Random Seed**: `42` across all Python, NumPy, and PyTorch RNGs

### Checkpoint Persistence & Resume Protocol:
- **Periodic Drive Sync**: `--backup-dir` ensures checkpoints are copied to Google Drive at the end of **every epoch**.
- **`FORCE_RETRAIN = False` (Default)**: Completed experiments are skipped. Interrupted experiments resume from `latest.pt`.
- **`FORCE_RETRAIN = True`**: Ignores existing checkpoints and initiates a fresh training run from epoch 1.


In [ ]:
import subprocess
import json
import time

# Set FORCE_RETRAIN = True only if you want to discard existing checkpoints and retrain from scratch
FORCE_RETRAIN = False

EXPERIMENT_MATRIX = [
    # --- LEVIR-CD EXPERIMENTS ---
    {
        "id": "exp1_levir_unet",
        "model": "siamese_unet",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp2_levir_vit",
        "model": "vit",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp3_levir_swin",
        "model": "swin",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    # --- OSCD EXPERIMENTS ---
    {
        "id": "exp4_oscd_unet",
        "model": "siamese_unet",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp5_oscd_vit",
        "model": "vit",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp6_oscd_swin",
        "model": "swin",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
]

manifest_path = DRIVE_OUTPUT_DIR / "experiment_manifest.json"
manifest = {}
if manifest_path.exists():
    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception:
        manifest = {}

print(f"Loaded existing manifest with {len(manifest)} recorded runs.")

for idx, exp in enumerate(EXPERIMENT_MATRIX, 1):
    exp_id = exp["id"]
    model_name = exp["model"]
    dataset_name = exp["dataset"]
    safe_ds = dataset_name.lower().replace(" ", "_")
    best_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_best.pt"
    latest_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_latest.pt"

    print(f"\n=======================================================")
    print(f"[{idx}/6] Experiment: {exp_id} ({model_name} on {dataset_name})")
    print(f"=======================================================")

    if not exp["enabled"]:
        print(f"  SKIPPED: Dataset for {dataset_name} is marked disabled/unavailable.")
        continue

    # Overwrite protection: Skip already completed experiments unless FORCE_RETRAIN is True
    if not FORCE_RETRAIN and best_ckpt_drive.exists() and manifest.get(exp_id, {}).get("status") == "COMPLETED":
        print(f"  SKIPPED: Experiment already completed with checkpoint: {best_ckpt_drive.name}")
        continue

    cmd = [
        sys.executable,
        "scripts/train_model.py",
        "--model", model_name,
        "--dataset", dataset_name,
        "--data-dir", exp["data_dir"],
        "--epochs", str(exp["epochs"]),
        "--batch-size", str(exp["batch_size"]),
        "--patch-size", "256",
        "--lr", "3e-4",
        "--loss", "bce_dice",
        "--optimizer", "adamw",
        "--scheduler", "cosine",
        "--device", "cuda",
        "--seed", "42",
        "--checkpoint-dir", "models/checkpoints",
        "--experiments-dir", "experiments",
        "--backup-dir", str(DRIVE_CHECKPOINTS_DIR),
    ]

    # Explicit Resume vs. Fresh Retrain Logic
    if FORCE_RETRAIN:
        print("  [FORCE_RETRAIN=True] Starting fresh experiment from epoch 1. Existing latest checkpoint will not be resumed.")
    else:
        if latest_ckpt_drive.exists():
            print(f"  Found previous run checkpoint! Resuming from: {latest_ckpt_drive.name}")
            cmd.extend(["--resume", str(latest_ckpt_drive)])

    if exp["bands"]:
        cmd.append("--bands")
        cmd.extend(exp["bands"])

    print(f"  Launching training command: {' '.join(cmd[:6])} ...")
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, check=True)
        duration_sec = round(time.time() - t0, 2)
        local_best = Path("models/checkpoints") / f"{model_name}_{safe_ds}_best.pt"
        
        if local_best.exists():
            shutil.copy2(local_best, best_ckpt_drive)
            manifest[exp_id] = {
                "status": "COMPLETED",
                "model": model_name,
                "dataset": dataset_name,
                "checkpoint": str(best_ckpt_drive),
                "duration_sec": duration_sec,
                "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            print(f"  Experiment {exp_id} COMPLETED in {duration_sec}s. Checkpoint synced to Drive.")
        else:
            manifest[exp_id] = {"status": "FAILED", "error": "Best checkpoint not found after training"}
            print(f"  WARNING: Process exited cleanly but {local_best.name} was not created.")
    except subprocess.CalledProcessError as err:
        duration_sec = round(time.time() - t0, 2)
        manifest[exp_id] = {
            "status": "FAILED",
            "error": f"Process exited with code {err.returncode}",
            "duration_sec": duration_sec,
        }
        print(f"  ERROR: Experiment {exp_id} failed with exit code {err.returncode}. Continuing remaining runs.")
    except Exception as exc:
        manifest[exp_id] = {"status": "FAILED", "error": str(exc)}
        print(f"  ERROR: Unexpected failure in {exp_id}: {exc}. Continuing remaining runs.")

    # Save manifest snapshot to Google Drive immediately after each run
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("\n Training sequence finished. Manifest saved to Google Drive.")


## Step 8: Standardized Quantitative Benchmarking on Dedicated Test Split
Evaluates the three models under identical test conditions:
- **Safeguard**: Verifies that all 3 model checkpoints exist before benchmarking. If any checkpoint is missing, reports the missing architecture and aborts rather than producing an incomplete comparison.
- **Metrics**: Precision, Recall, F1-Score, IoU, Overall Accuracy, Kappa
- **Visuals**: Pixel-level Confusion Matrices & Qualitative Prediction Overlays
- **Outputs**: Saved directly to Google Drive (`DRIVE_EXPERIMENTS_DIR` and `DRIVE_VIZ_DIR`)


In [ ]:
if dataset_status["levir_ok"]:
    print("=== Checking Checkpoint Integrity for LEVIR-CD ===")
    required_models = ["siamese_unet", "vit", "swin"]
    missing_ckpts = []
    for m in required_models:
        ckpt_p = DRIVE_CHECKPOINTS_DIR / f"{m}_levir-cd_best.pt"
        if not ckpt_p.exists():
            missing_ckpts.append(ckpt_p.name)
            
    if missing_ckpts:
        print(f" ERROR: Cannot run fair 3-architecture benchmark for LEVIR-CD.")
        print(f"   Missing trained checkpoints: {missing_ckpts}")
        print("   Please train the missing models first. Aborting LEVIR-CD benchmark to prevent false completion.")
    else:
        print(" All 3 model checkpoints found. Running LEVIR-CD standardized benchmark on TEST split...")
        subprocess.run([
            sys.executable,
            "scripts/evaluate_model.py",
            "--dataset", "LEVIR-CD",
            "--data-dir", str(LEVIR_CD_DIR),
            "--split", "test",
            "--patch-size", "256",
            "--batch-size", "8",
            "--threshold", "0.5",
            "--device", "cuda",
            "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
            "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
            "--viz-dir", str(DRIVE_VIZ_DIR),
            "--save-visuals",
            "--visual-samples", "6",
        ], check=True)


In [ ]:
if dataset_status["oscd_ok"]:
    print("=== Checking Checkpoint Integrity for OSCD ===")
    required_models = ["siamese_unet", "vit", "swin"]
    missing_ckpts = []
    for m in required_models:
        ckpt_p = DRIVE_CHECKPOINTS_DIR / f"{m}_oscd_best.pt"
        if not ckpt_p.exists():
            missing_ckpts.append(ckpt_p.name)
            
    if missing_ckpts:
        print(f" ERROR: Cannot run fair 3-architecture benchmark for OSCD.")
        print(f"   Missing trained checkpoints: {missing_ckpts}")
        print("   Please train the missing models first. Aborting OSCD benchmark to prevent false completion.")
    else:
        print(" All 3 model checkpoints found. Running OSCD standardized benchmark on TEST split...")
        subprocess.run([
            sys.executable,
            "scripts/evaluate_model.py",
            "--dataset", "OSCD",
            "--data-dir", str(OSCD_DIR),
            "--split", "test",
            "--patch-size", "256",
            "--batch-size", "8",
            "--threshold", "0.5",
            "--device", "cuda",
            "--bands", "B04", "B03", "B02",
            "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
            "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
            "--viz-dir", str(DRIVE_VIZ_DIR),
            "--save-visuals",
            "--visual-samples", "6",
        ], check=True)


## Step 9: Render Model Comparison Charts & Confusion Matrices


In [ ]:
import glob
from IPython.display import Image, display

# 1. Comparison bar charts from Drive
charts = sorted(glob.glob(str(DRIVE_VIZ_DIR / "*_comparison.png")))
if charts:
    print(f"Found {len(charts)} comparison charts:")
    for c in charts:
        print(f"Displaying: {c}")
        display(Image(filename=c))
else:
    print("No comparison charts found in visualizations folder.")

# 2. Confusion Matrices from Drive
cms = sorted(glob.glob(str(DRIVE_VIZ_DIR / "*_cm.png")))
if cms:
    print(f"\nFound {len(cms)} confusion matrices:")
    for cm in cms:
        display(Image(filename=cm))


## Step 10: Final Google Drive Backup Audit & Verification
Audits the persistent Google Drive directories to confirm that all trained checkpoints, evaluation summaries, confusion matrices, and prediction samples exist with valid non-zero file sizes.


In [ ]:
import os
import time

print("=== AUDIT: Verified Persistent Files on Google Drive ===")

def audit_dir(name: str, path: Path):
    print(f"\n[{name}] -> {path}")
    if not path.is_dir():
        print("  Directory does not exist!")
        return
    files = sorted(path.rglob("*.*"))
    if not files:
        print("  (Empty directory)")
        return
    for f in files:
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            rel_name = str(f.relative_to(path))
            mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(f.stat().st_mtime))
            print(f"   {rel_name:<50} | {size_mb:>8.2f} MB | {mtime}")

audit_dir("Checkpoints", DRIVE_CHECKPOINTS_DIR)
audit_dir("Experiments & Benchmarks", DRIVE_EXPERIMENTS_DIR)
audit_dir("Visualizations & Predictions", DRIVE_VIZ_DIR)

print("\n Audit completed. All experimental assets are safely preserved on Google Drive.")
